# The run: actor-critic PPO on Qwen2.5-0.5B-Instruct

[`01_actor_critic_and_gae.ipynb`](01_actor_critic_and_gae.ipynb) checked the mathematics
without a GPU. This one spends one. Same frozen data, same deterministic reward, same model
and the same clipped surrogate as
[`../../membrane_grpo/grpo_scratch.py`](../../membrane_grpo/grpo_scratch.py) — the baseline
is a learned $V_\phi(s_t)$ instead of a group mean, and the advantage is per token instead
of per sequence.

**Scale: this is a smoke test.** A short run on one card, not a converged training run.
Every claim below is scoped to that, exactly as `membrane_grpo`'s own README scopes its.

---

## Before running: the card has to be free

`anton` has a single RTX 5070 Ti, 16 GiB, and the vLLM 9B service normally holds ~14.6 GiB
of it. Nothing here will fit alongside it. Stopping the stack needs `sudo`, there is no
`NOPASSWD` rule on this machine, and `deploy/stack.sh` says so in its own header — so it is
**run by a person at a terminal, not by this notebook and not by an agent**:

```sh
deploy/stack.sh down                       # if you are on a branch that has it
sudo systemctl stop vllm-qwen membraneclaw-agent   # equivalent, any branch
```

and afterwards:

```sh
deploy/stack.sh up
```

The next cell refuses to continue rather than letting a run OOM twenty minutes in.

In [ ]:
import sys, json, subprocess
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "ppo_ac.py").exists() else Path("experiments/notebooks/smoke_test")
sys.path.insert(0, str(HERE.resolve()))

import torch
import ppo_ac
from ppo_ac import Config, train, resolve_device

MIN_FREE_MIB = 8000
BLOCKING_UNITS = ["vllm-qwen", "membraneclaw-agent"]

assert torch.cuda.is_available(), "no CUDA device; this notebook is the GPU half of the pair"
name = torch.cuda.get_device_name(0)

# Two sources, because on this machine they disagree and only one of them is right.
# `torch.cuda.mem_get_info` reported 13454 MiB free while nvidia-smi reported 544 --
# under WSL2 the CUDA driver counts memory it could page to host RAM as available to a
# new context. Trusting it starts a run that either crawls or dies twenty minutes in.
# nvidia-smi's number is the one that matches the card.
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True,
)
used, total = (int(x) for x in smi.stdout.strip().split(", "))
torch_free = torch.cuda.mem_get_info()[0] // 2**20
free = total - used

print(f"{name}: {free} / {total} MiB free per nvidia-smi   (torch claims {torch_free}; "
      f"see the comment above for why they differ)")

busy = {u: subprocess.run(["systemctl", "is-active", u], capture_output=True, text=True).stdout.strip()
        for u in BLOCKING_UNITS}
running = [u for u, s in busy.items() if s == "active"]

if free < MIN_FREE_MIB or running:
    raise RuntimeError(
        f"the card is not free: {free} MiB available (need ~{MIN_FREE_MIB}), "
        f"still running {running or 'nothing'}.\n"
        f"From a terminal (needs sudo, no NOPASSWD rule here -- not from this notebook):\n"
        f"    sudo systemctl stop {' '.join(BLOCKING_UNITS)}\n"
        f"and afterwards:  sudo systemctl start {' '.join(BLOCKING_UNITS)}"
    )
print("card is free; the run may proceed")

## The configuration, and why it is what it is

Sized against `grpo_scratch`'s reference config so the two runs can be read on one axis.
GRPO's step is 4 prompts × group of 8 = **32 sequences**; this is 8 prompts × 1 sample =
**8 sequences**, because the algorithm does not need a second sample of the same prompt to
form a baseline. That is the saving the critic buys, and it is the reason `steps` can be
larger for the same wall clock.

`probe_throughput.py` measured 17.5 s/step for GRPO's 32 sequences on this card, so 8
sequences should land near 5 s/step and 200 steps in roughly 20 minutes.

In [ ]:
cfg = Config(
    steps=200,
    prompts_per_step=8,
    samples_per_prompt=1,     # the point: no group needed
    max_new_tokens=640,
    temperature=1.0,
    lr=1e-5,
    value_lr=5e-6,            # derived in 01; ||h||_1 ~ 4.3e3 on this model
    lam=1.0,                  # A_t = R - V(s_t); the honest default
    gamma=1.0,
    clip_eps=0.2,
    value_clip_eps=0.2,
    value_coef=0.5,
    value_detach=True,        # the critic is a probe, not a co-author of the trunk
    whiten_advantages=False,  # see 01: whitening would hide whether the critic works
    inner_epochs=1,           # clip inert by construction; comparable to the GRPO run
    micro_batch=2,
    lora_r=16,
    weights="MAIN",
    seed=0,
    dtype="bfloat16",
    split="train",
)
OUT = HERE / "runs" / f"ppo-ac-main-lam{cfg.lam}-s{cfg.seed}"
print(json.dumps({k: v for k, v in cfg.__dict__.items() if k != "metrics"}, indent=2))

In [ ]:
history = train(cfg, OUT, resolve_device("auto"))
print(f"\n{len(history)} steps -> {OUT}")

## Curves

Five things, and the fifth is the one this experiment exists to see.

In [ ]:
import matplotlib.pyplot as plt

def load(p):
    p = Path(p)
    return [json.loads(l) for l in p.read_text().splitlines()] if p.exists() else []

rows = history if "history" in dir() and history else load(OUT / "metrics.jsonl")
step = [r["step"] for r in rows]

def smooth(xs, k=10):
    return [sum(xs[max(0, i-k+1):i+1]) / len(xs[max(0, i-k+1):i+1]) for i in range(len(xs))]

fig, ax = plt.subplots(2, 3, figsize=(15, 7))
BASELINE = 0.086   # runs/baseline-0.5b-v2/eval_dev_greedy.json -> overall.reward

ax[0,0].plot(step, [r["reward_mean"] for r in rows], alpha=.25, color="C0")
ax[0,0].plot(step, smooth([r["reward_mean"] for r in rows]), color="C0", lw=2)
ax[0,0].axhline(BASELINE, ls="--", color="k", lw=1, label="frozen 0.5B (greedy, dev)")
ax[0,0].set_title("reward"); ax[0,0].legend(fontsize=8)

ax[0,1].plot(step, [r["value_mean"] for r in rows], label="V")
ax[0,1].plot(step, smooth([r["reward_mean"] for r in rows]), label="reward (smoothed)", ls=":")
ax[0,1].set_title("what the critic believes vs. what happened"); ax[0,1].legend(fontsize=8)

ax[0,2].plot(step, [r["value_mae"] for r in rows], color="C3")
ax[0,2].set_title("value MAE  (flat = predicting the mean)")

ax[1,0].plot(step, [r["adv_std"] for r in rows], color="C2")
ax[1,0].set_title("advantage spread  (0 = no gradient)")

ax[1,1].plot(step, [r["completion_tokens"] for r in rows], color="C4")
ax[1,1].axhline(108, ls="--", color="k", lw=1, label="frozen baseline: 108")
ax[1,1].set_title("completion length"); ax[1,1].legend(fontsize=8)

ax[1,2].plot(step, [r["adv_zero_frac"] for r in rows], color="C1", label="actor-critic")
ax[1,2].axhline(0.16, ls="--", color="k", lw=1, label="GRPO's degenerate-group rate")
ax[1,2].set_ylim(-0.02, 1.0)
ax[1,2].set_title("fraction of tokens with zero advantage"); ax[1,2].legend(fontsize=8)

for a in ax.ravel():
    a.set_xlabel("step"); a.grid(alpha=.3)
fig.tight_layout()
plt.show()

## Read against GRPO

The comparison that matters is not "which reward is higher" — one short run each settles
nothing about that. It is the **structural** claim from 01, which a run can confirm or
refute: GRPO throws away 16% of its groups; a critic throws away nothing.

In [ ]:
grpo_rows = load(ppo_ac.GRPO_DIR / "runs" / "smoke-grpo-cpu" / "metrics.jsonl")
for label, rs in (("GRPO (CPU smoke, 3 steps)", grpo_rows),
                  ("actor-critic PPO (this run)", rows)):
    if not rs:
        continue
    zero_grad = sum(1 for r in rs if r["grad_norm"] == 0.0)
    print(f"{label}")
    print(f"   steps                {len(rs)}")
    print(f"   steps with |grad|=0  {zero_grad}  ({zero_grad/len(rs):.0%})")
    print(f"   mean reward          {sum(r['reward_mean'] for r in rs)/len(rs):.4f}")
    print(f"   final reward         {rs[-1]['reward_mean']:.4f}")
    print(f"   sequences per step   {rs[0].get('unique_completions', float('nan'))}")
    print()

## The ablation worth running next

$\lambda$ is the one dial that has no counterpart in GRPO at all — it is what per-token
credit assignment *means*. `--lam 1.0` is $A_t = R - V(s_t)$, unbiased and noisy;
`--lam 0.0` is pure bootstrapping off the critic. Same seed, same data, one flag:

```sh
.venv/bin/python ppo_ac.py --lam 0.0 --out runs/ppo-ac-main-lam0.0-s0
.venv/bin/python ppo_ac.py --whiten-advantages 1 --out runs/ppo-ac-whitened-s0
.venv/bin/python ppo_ac.py --value-init-bias 0 --out runs/ppo-ac-cold-critic-s0
```

The third one is expected to fail, and to fail in the specific way 01 predicts: a critic
starting at zero cannot produce a negative advantage against a non-negative reward, so the
first updates reinforce every completion including the worthless ones. If it does not fail
that way, the argument in 01 is wrong and should be corrected here.

## What a green run here would and would not show

**Would**: that the loop is correct end to end on real hardware, that a learned baseline
produces a usable gradient on every step including the ones GRPO discards, and how fast the
critic converges to something better than a constant.

**Would not**: that actor-critic beats GRPO on this task. That needs both runs at the same
budget on the same split with the sealed `test.jsonl` still sealed, and it is not what a
smoke test is for. The frozen baseline's own numbers are the reminder of how far there is
to go — `cause_acc` 0.145 against a 1/7 = 0.143 chance rate, and `pass@8` of exactly zero
across 1,600 samples.